### Imports

In [217]:
import os, shutil
import time
import sqlite3
import csv
from psutil import virtual_memory
import pandas as pd

In [218]:
def mem():
    print(f'used memory : {round(virtual_memory()[3]/(1024*1024*1024)*10)/10}Go')

In [219]:
def stats(): 
    print("--- %s seconds ---" % (time.time() - start_time))
    mem()

### Path set-up

In [220]:
if "DATA_DIR" not in locals():
    DATA_DIR = "./data/"
else:
    print(DATA_DIR)

if os.path.exists(DATA_DIR) and os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)

./data/


In [221]:
if "OUTPUT_DATA_FOLDER" not in locals():
    OUTPUT_DATA_FOLDER = "./output/"
else:
    print(OUTPUT_DATA_FOLDER)

if os.path.exists(OUTPUT_DATA_FOLDER) and os.path.isdir(OUTPUT_DATA_FOLDER):
    shutil.rmtree(OUTPUT_DATA_FOLDER)
os.makedirs(os.path.dirname(OUTPUT_DATA_FOLDER), exist_ok=True)

./output/


## SQLITE


In [222]:
!rm sirene.db

In [223]:
connection = sqlite3.connect('sirene.db')

In [224]:
cursor = connection.cursor()

## Unité Légale

In [225]:
cursor.execute('''CREATE TABLE IF NOT EXISTS unite_legale
               (siren,
                date_creation_unite_legale,
                sigle,
                prenom,
                identifiant_association_unite_legale,
                tranche_effectif_salarie_unite_legale,
                date_mise_a_jour_unite_legale,
                categorie_entreprise,
                etat_administratif_unite_legale,
                nom,
                nom_usage,
                nom_raison_sociale,
                nature_juridique_unite_legale,
                activite_principale_unite_legale,
                economie_sociale_solidaire_unite_legale)
                ''')

In [226]:
cursor.execute('''
                CREATE UNIQUE INDEX index_siren
                ON unite_legale (siren);
                ''')

In [227]:
connection.commit()

In [228]:
start_time = time.time()

df_unite_legale = pd.read_csv(
    "https://files.data.gouv.fr/insee-sirene/StockUniteLegale_utf8.zip",
    compression="zip",
    dtype=str,
    usecols=[
        "siren",
        "dateCreationUniteLegale",
        "sigleUniteLegale",
        "prenom1UniteLegale",
        "identifiantAssociationUniteLegale",
        "trancheEffectifsUniteLegale",
        "dateDernierTraitementUniteLegale",
        "categorieEntreprise",
        "etatAdministratifUniteLegale",
        "nomUniteLegale",
        "nomUsageUniteLegale",
        "denominationUniteLegale",
        "categorieJuridiqueUniteLegale",
        "activitePrincipaleUniteLegale",
        "economieSocialeSolidaireUniteLegale",
    ],
)
# Rename columns
df_unite_legale = df_unite_legale.rename(
    columns={
        "dateCreationUniteLegale": "date_creation_unite_legale",
        "sigleUniteLegale": "sigle",
        "prenom1UniteLegale": "prenom",
        "trancheEffectifsUniteLegale": "tranche_effectif_salarie_unite_legale",
        "dateDernierTraitementUniteLegale": "date_mise_a_jour_unite_legale",
        "categorieEntreprise": "categorie_entreprise",
        "etatAdministratifUniteLegale":"etat_administratif_unite_legale",
        "nomUniteLegale": "nom",
        "nomUsageUniteLegale": "nom_usage",
        "denominationUniteLegale": "nom_raison_sociale",
        "categorieJuridiqueUniteLegale": "nature_juridique_unite_legale",
        "activitePrincipaleUniteLegale": "activite_principale_unite_legale",
        "economieSocialeSolidaireUniteLegale":"economie_sociale_solidaire_unite_legale",
        "identifiantAssociationUniteLegale":"identifiant_association_unite_legale",
    }
)
stats()

--- 559.2660949230194 seconds ---
used memory : 15.6Go


In [229]:
start_time = time.time()
df_unite_legale.to_sql("unite_legale", connection, if_exists='append', index=False)
stats()

--- 189.82977485656738 seconds ---
used memory : 14.5Go


In [230]:
for row in cursor.execute('SELECT * FROM unite_legale LIMIT 10;'):
    print(row)

('000325175', '2000-09-26', None, 'THIERRY', None, None, '2019-12-13T13:21:28', 'PME', 'A', 'JANOYER', None, None, '1000', '32.12Z', None)
('001807254', '1972-05-01', None, 'JACQUES-LUCIEN', None, None, '2016-07-10T05:00:06', None, 'C', 'BRETON', None, None, '1000', '85.59A', None)
('005410220', '1954-12-25', None, 'GEORGES', None, None, None, None, 'C', 'WATTEBLED', None, None, '1000', '22.02', None)
('005410345', None, None, 'MICHEL', None, None, None, None, 'C', 'DEBRAY', None, None, '1000', '79.06', None)
('005410394', '1954-12-25', None, 'ROBERT', None, None, None, None, 'C', 'DAULT', None, None, '1000', '64.42', None)
('005410428', '1954-01-01', None, 'RENE', None, None, None, None, 'C', 'DINGEON', None, None, '1000', '70.2C', None)
('005410436', None, None, 'MARCEL', None, None, None, None, 'C', 'CARBONNET', None, None, '1000', '57.11', None)
('005410485', None, None, 'RENE', None, None, None, None, 'C', 'LECRIVAIN', None, None, '1000', '64.42', None)
('005410493', '1954-12-25',

### Add nom_complet

In [231]:
def nom_complet(nature_juridique_unite_legale, sigle, prenom, nom, nom_usage, nom_raison_sociale):
    if nature_juridique_unite_legale == "1000":
        if sigle is not None:
            if (prenom is not None) & (nom is not None):
                if(nom_usage is not None):
                    return (
                        prenom.lower()
                        + " "
                        + nom_usage.lower()
                        + " ("
                        + nom.lower()
                        + ", "
                        + sigle.lower()
                        + ")"
                    )
                else:
                    return (
                        prenom.lower()
                        + " "
                        + nom.lower()
                        + " ("
                        + sigle.lower()
                        + ")"
                    )
            else:
                return None
        else:
            if (prenom is not None) & (nom is not None):
                if(nom_usage is not None):
                    return (
                        prenom.lower()
                        + " "
                        + nom_usage.lower()
                        + " ("
                        + nom.lower()
                        + ")"
                    )
                else:
                    return prenom.lower() + " " + nom.lower()
            else:
                return None
    else:
        if(sigle is not None):
            if(nom_raison_sociale is not None):
                return nom_raison_sociale.lower() + " (" + sigle.lower() + ")"
            else:
                return None
        else:
            if(nom_raison_sociale is not None):
                return nom_raison_sociale.lower()
            else:
                return None

In [232]:
connection.create_function("nom_complet", 6, nom_complet)

In [233]:
connection.commit()

In [278]:
for row in cursor.execute('''
    SELECT 
        siren, 
        nom_complet(
            nature_juridique_unite_legale, 
            sigle, 
            prenom, 
            nom, 
            nom_usage, 
            nom_raison_sociale
        )
    FROM
        unite_legale 
    LIMIT 10;
'''):
    print(row)

('000325175', 'thierry janoyer')
('001807254', 'jacques-lucien breton')
('005410220', 'georges wattebled')
('005410345', 'michel debray')
('005410394', 'robert dault')
('005410428', 'rene dingeon')
('005410436', 'marcel carbonnet')
('005410485', 'rene lecrivain')
('005410493', 'pierre godin')
('005410501', 'andree vandamme (carpentier)')


### Add entrepreneur individuel and section activité principale

In [237]:
def is_ei(nature_juridique_unite_legale):
    if(nature_juridique_unite_legale in ["1", "10", "1000"]):
        return 1
    else:
        return 0


In [238]:
connection.create_function("is_ei", 1, is_ei)

In [279]:
for row in cursor.execute('''
    SELECT 
        siren, 
        is_ei(
            nature_juridique_unite_legale
        )
    FROM
        unite_legale 
    LIMIT 10;
'''):
    print(row)

('000325175', 1)
('001807254', 1)
('005410220', 1)
('005410345', 1)
('005410394', 1)
('005410428', 1)
('005410436', 1)
('005410485', 1)
('005410493', 1)
('005410501', 1)


In [239]:
sections_NAF = {
"01":"A","02":"A","03":"A","05":"B","06":"B","07":"B","08":"B","09":"B","10":"C","11":"C","12":"C","13":"C","14":"C",
 "15":"C","16":"C","17":"C","18":"C","19":"C","20":"C","21":"C","22":"C","23":"C","24":"C","25":"C","26":"C","27":"C",
 "28":"C","29":"C","30":"C","31":"C","32":"C","33":"C","35":"D","36":"E","37":"E","38":"E","39":"E","41":"F","42":"F",
 "43":"F","45":"G","46":"G","47":"G","49":"H","50":"H","51":"H","52":"H","53":"H","55":"I","56":"I","58":"J","59":"J",
 "60":"J","61":"J","62":"J","63":"J","64":"K","65":"K","66":"K","68":"L","69":"M","70":"M","71":"M","72":"M","73":"M",
 "74":"M","75":"M","77":"N","78":"N","79":"N","80":"N","81":"N","82":"N","84":"O","85":"P","86":"Q","87":"Q","88":"Q",
 "90":"R","91":"R","92":"R","93":"R","94":"S","95":"S","96":"S","97":"T","98":"T","99":"U"
}

In [280]:
def get_section(activite_principale_unite_legale):
    code_naf = activite_principale_unite_legale[:2]
    section_activite_principale = sections_NAF[code_naf] if code_naf in sections_NAF else None
    return section_activite_principale

In [281]:
connection.create_function("get_section", 1, get_section)

In [282]:
for row in cursor.execute('''
    SELECT 
        siren, 
        activite_principale_unite_legale,
        get_section(
            activite_principale_unite_legale
        )
    FROM
        unite_legale 
    LIMIT 10;
'''):
    print(row)

('000325175', '32.12Z', 'C')
('001807254', '85.59A', 'P')
('005410220', '22.02', 'C')
('005410345', '79.06', 'N')
('005410394', '64.42', 'K')
('005410428', '70.2C', 'M')
('005410436', '57.11', None)
('005410485', '64.42', 'K')
('005410493', '86.06', 'Q')
('005410501', '55.4B', 'I')


## Établissements

In [313]:
# Create list of departement zip codes
all_deps = [
    *"-0".join(list(str(x) for x in range(0, 10))).split("-")[1:],
    *list(str(x) for x in range(10, 20)),
    *["2A", "2B"],
    *list(str(x) for x in range(21, 96)),
    *"-7510".join(list(str(x) for x in range(0, 10))).split("-")[1:],
    *"-751".join(list(str(x) for x in range(10, 21))).split("-")[1:],
    *["971", "972", "973", "974", "976"],
    *[""],
]
# Remove Paris zip code
all_deps.remove("75")

In [314]:
#all_deps = ["23"]

In [ ]:
%%time
# Upload geo data by departement
for dep in all_deps:
    start_time = time.time()
    url = "https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_" + dep + ".csv.gz"
    print(url)
    df_dep = pd.read_csv(
        url,
        compression="gzip",
        dtype=str,
        usecols=[
            "siren",
            "siret",
            "dateCreationEtablissement",
            "trancheEffectifsEtablissement",
            "activitePrincipaleRegistreMetiersEtablissement",
            "etablissementSiege",
            "numeroVoieEtablissement",
            "libelleVoieEtablissement",
            "codePostalEtablissement",
            "libelleCommuneEtablissement",
            "libelleCedexEtablissement",
            "typeVoieEtablissement",
            "codeCommuneEtablissement",
            "codeCedexEtablissement",
            "complementAdresseEtablissement",
            "distributionSpecialeEtablissement",
            "complementAdresse2Etablissement",
            "indiceRepetition2Etablissement",
            "libelleCedex2Etablissement",
            "codeCedex2Etablissement",
            "numeroVoie2Etablissement",
            "typeVoie2Etablissement",
            "libelleVoie2Etablissement",
            "codeCommune2Etablissement",
            "libelleCommune2Etablissement",
            "distributionSpeciale2Etablissement",
            "dateDebut",
            "etatAdministratifEtablissement",
            "enseigne1Etablissement",
            "enseigne1Etablissement",
            "enseigne2Etablissement",
            "enseigne3Etablissement",
            "denominationUsuelleEtablissement",
            "activitePrincipaleEtablissement",
            "geo_adresse",
            "geo_id",
            "longitude",
            "latitude",
            "indiceRepetitionEtablissement",
            "libelleCommuneEtrangerEtablissement",
            "codePaysEtrangerEtablissement",
            "libellePaysEtrangerEtablissement",
            "libelleCommuneEtranger2Etablissement",
            "codePaysEtranger2Etablissement",
            "libellePaysEtranger2Etablissement",
        ],
    )
    df_dep = df_dep.rename(
        columns={
            "dateCreationEtablissement": "date_creation",
            "trancheEffectifsEtablissement": "tranche_effectif_salarie",
            "activitePrincipaleRegistreMetiersEtablissement": "activite_principale_registre_metier",
            "etablissementSiege": "is_siege",
            "numeroVoieEtablissement": "numero_voie",
            "typeVoieEtablissement": "type_voie",
            "libelleVoieEtablissement": "libelle_voie",
            "codePostalEtablissement": "code_postal",
            "libelleCedexEtablissement": "libelle_cedex",
            "libelleCommuneEtablissement": "libelle_commune",
            "codeCommuneEtablissement": "commune",
            "complementAdresseEtablissement": "complement_adresse",
            "complementAdresse2Etablissement": "complement_adresse_2",
            "numeroVoie2Etablissement": "numero_voie_2",
            "indiceRepetition2Etablissement": "indice_repetition_2",
            "typeVoie2Etablissement": "type_voie_2",
            "libelleVoie2Etablissement": "libelle_voie_2",
            "codeCommune2Etablissement": "commune_2",
            "libelleCommune2Etablissement": "libelle_commune_2",
            "codeCedex2Etablissement": "cedex_2",
            "libelleCedex2Etablissement": "libelle_cedex_2",
            "codeCedexEtablissement": "cedex",
            "dateDebut": "date_debut_activite",
            "distributionSpecialeEtablissement": "distribution_speciale",
            "distributionSpeciale2Etablissement": "distribution_speciale_2",
            "etatAdministratifEtablissement": "etat_administratif_etablissement",
            "enseigne1Etablissement": "enseigne_1",
            "enseigne2Etablissement": "enseigne_2",
            "enseigne3Etablissement": "enseigne_3",
            "activitePrincipaleEtablissement": "activite_principale",
            "indiceRepetitionEtablissement": "indice_repetition",
            "denominationUsuelleEtablissement": "nom_commercial",
            "libelleCommuneEtrangerEtablissement": "libelle_commune_etranger",
            "codePaysEtrangerEtablissement": "code_pays_etranger",
            "libellePaysEtrangerEtablissement": "libelle_pays_etranger",
            "libelleCommuneEtranger2Etablissement": "libelle_commune_etranger_2",
            "codePaysEtranger2Etablissement": "code_pays_etranger_2",
            "libellePaysEtranger2Etablissement": "libelle_pays_etranger_2",
        }
    )
    stats()
    cursor.execute(f'''DROP TABLE IF EXISTS {f"siret_{dep}"}''')
    cursor.execute(f'''CREATE TABLE IF NOT EXISTS {f"siret_{dep}"}
            (siren,
            siret,
            date_creation,
            tranche_effectif_salarie,
            activite_principale_registre_metier,
            is_siege,
            numero_voie,
            type_voie,
            libelle_voie,
            code_postal,
            libelle_cedex,
            libelle_commune,
            commune,
            complement_adresse,
            complement_adresse_2,
            numero_voie_2,
            indice_repetition_2,
            type_voie_2,
            libelle_voie_2,
            commune_2,
            libelle_commune_2,
            cedex_2,
            libelle_cedex_2,
            cedex,
            date_debut_activite,
            distribution_speciale,
            distribution_speciale_2,
            etat_administratif_etablissement,
            enseigne_1,
            enseigne_2,
            enseigne_3,
            activite_principale,
            indice_repetition,
            nom_commercial,
            libelle_commune_etranger,
            code_pays_etranger,
            libelle_pays_etranger,
            libelle_commune_etranger_2,
            code_pays_etranger_2,
            libelle_pays_etranger_2,
            longitude,
            latitude,
            geo_adresse,
            geo_id)
            ''')
    
    start_time = time.time()
    df_dep.to_sql(f"siret_{dep}", connection, if_exists='append', index=False)
    connection.commit()
    stats()

https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_01.csv.gz
--- 15.19149899482727 seconds ---
used memory : 16.9Go
--- 5.025361776351929 seconds ---
used memory : 16.9Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_02.csv.gz
--- 11.618398904800415 seconds ---
used memory : 16.9Go
--- 3.789867877960205 seconds ---
used memory : 16.9Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_03.csv.gz
--- 9.422542095184326 seconds ---
used memory : 16.9Go
--- 3.1476171016693115 seconds ---
used memory : 16.9Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_04.csv.gz
--- 6.285191059112549 seconds ---
used memory : 16.9Go
--- 2.2290570735931396 seconds ---
used memory : 16.9Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_05.csv.gz
--- 6.309167146682739 seconds ---
used memory : 16.8Go
--- 2.0479490756988525 seconds ---
used memory : 16.8Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_06.csv.gz


In [285]:
for row in cursor.execute('SELECT * FROM siret_23 LIMIT 2;'):
    print(row)

('038822102', '03882210200018', '1997-12-25', None, None, 'true', None, None, None, '23460', None, 'LE MONTEIL-AU-VICOMTE', '23134', None, None, None, None, None, None, None, None, None, None, None, '2008-01-01', None, None, 'A', None, None, None, '81.10Z', None, None, None, None, None, None, None, None, None, None, None, None)
('039016357', '03901635700012', None, None, None, 'true', None, None, None, '23700', None, 'AUZANCES', '23013', None, None, None, None, None, None, None, None, None, None, None, '1997-12-31', None, None, 'F', None, None, None, '70.3C', None, None, None, None, None, None, None, None, None, None, None, None)


### Create Adresse Compléte

In [286]:
def adresse_complete(complement_adresse, numero_voie, indice_repetition, type_voie, libelle_voie, libelle_commune, libelle_cedex, distribution_speciale, commune, cedex, libelle_commune_etranger, libelle_pays_etranger):    
    col_list = [complement_adresse, numero_voie, indice_repetition, type_voie, libelle_voie, distribution_speciale]
    adresse = ""
    for column in col_list:
        if(column is not None):
            adresse = adresse + " " + column if column is not None else ""
    if cedex is None:
        if commune is None:
            adresse =  adresse
        else:
            adresse = adresse + " " + commune + " " + libelle_commune
    else:
        adresse = adresse + " " + cedex + " " + libelle_cedex
    etranger_list = [libelle_commune_etranger, libelle_pays_etranger]
    for column in etranger_list:
        if(column is not None):
            adresse = adresse + " " + column if column is not None else ""
    return adresse.strip()

In [287]:
connection.create_function("adresse_complete", 12, adresse_complete)

In [304]:
for row in cursor.execute('''
    SELECT 
        siren, 
        adresse_complete(
            complement_adresse,
            numero_voie,
            indice_repetition,
            type_voie,
            libelle_voie,
            libelle_commune,
            libelle_cedex,
            distribution_speciale,
            commune,
            cedex,
            libelle_commune_etranger,
            libelle_pays_etranger
        ),
        adresse_complete(
            complement_adresse_2,
            numero_voie_2,
            indice_repetition_2,
            type_voie_2,
            libelle_voie_2,
            libelle_commune_2,
            libelle_cedex_2,
            distribution_speciale_2,
            commune_2,
            cedex_2,
            libelle_commune_etranger_2,
            libelle_pays_etranger_2
        )
    FROM
        siret_23 
    WHERE 
        libelle_voie_2 is not NULL
    LIMIT 10;
'''):
    print(row)

('172302119', '5 RUE MARTIN NADAUD 23031 BOUSSAC', '24 P GAMBETTA 23031 BOUSSAC')
('192300127', '4 RUE M BALANDIER 23061 CHENERAILLES', 'RTE D AHUN 23061 CHENERAILLES')
('309474625', '5 RUE DE VERDUN 23096 GUERET', '2 RUE DES TANNERIES 23096 GUERET')
('313041816', 'LES VARENNES CHER DU PRAT BP 231 23005 GUERET CEDEX', 'AV RENE CASSIN 23096 GUERET')
('315549352', '2 AV DU BERRY 23096 GUERET', '27 AV CHARLES DE GAULLE 23096 GUERET')
('316901453', '4 RUE GRANDE 23008 AUBUSSON', 'RUE PARDOUX DUPRAT 23008 AUBUSSON')
('322464686', 'CENTRE COMMERCIAL RECORD 46 AV D AUVERGNE 23096 GUERET', 'RUE ALEXANDRE GUILLON 23096 GUERET')
('325178432', 'RUE DES TOURS DE L HORLOGE 23079 FELLETIN', 'DOMICILE PERSONNEL RUE CHANTELOUBE 23079 FELLETIN')
('326191947', '1 PL BONNYAUD 23096 GUERET', '2 AV DE LA SENATORERIE 23096 GUERET')
('342404399', 'LE MAUPUY 23208 SAINT-LEGER-LE-GUERETOIS', 'RAVIERES DE VASSIVINS ROCHE BLANCHE 23208 SAINT-LEGER-LE-GUERETOIS')


### Add dep/coord columns

In [292]:
def get_departement(commune):
    departement = str(commune)[:3] if str(commune)[:2]== "97" else (None if commune is None else str(commune)[:2])
    return departement

In [293]:
def get_coordonnees(longitude, latitude):
    coordonnees = None if (longitude is None) or (latitude is None) else f"{latitude},{longitude}"
    return coordonnees

In [294]:
connection.create_function("get_coordonnees", 2, get_coordonnees)
connection.create_function("get_departement", 1, get_departement)

In [295]:
for row in cursor.execute('''
    SELECT 
        siren, 
        get_coordonnees(
            longitude, 
            latitude
        ),
        get_departement(
            commune
        )
    FROM
        siret_23 
    LIMIT 10;
'''):
    print(row)

('038822102', None, '23')
('039016357', None, '23')
('039027305', '45.981419,2.292848', '23')
('039315312', '46.30549,1.66628', '23')
('056810336', '46.168938,1.874098', '23')
('061200580', '46.171967,1.869143', '23')
('061200580', '46.172173,1.868873', '23')
('061200580', '46.185918,1.880452', '23')
('067800425', '46.177336,1.871078', '23')
('067800425', '46.169271,1.870178', '23')


### Nombre d'établissements

In [296]:
cursor.execute(f'''DROP TABLE IF EXISTS count_etab''')

In [297]:
cursor.execute('''CREATE TABLE count_etab (siren VARCHAR(10), count INTEGER)''')

### Add columns

In [298]:
def get_enseignes(enseigne_1, enseigne_2, enseigne_3, nom_commercial):
    return ' '.join(list(filter(None,set([enseigne_1, enseigne_2, enseigne_3, nom_commercial]))))

In [299]:
connection.create_function("get_enseignes", 4, get_enseignes)

In [301]:
for row in cursor.execute('''
    SELECT
        siren, 
        enseigne_1, 
        get_enseignes(
            enseigne_1,
            enseigne_2,
            enseigne_3,
            nom_commercial
        )
    FROM
        siret_23
    WHERE 
        enseigne_3 is not NULL 
    LIMIT 10;
'''):
    print(row)

('430303909', 'MARECHALERIE VINCENT', 'CIE SUR MESURE CHEZ VINCE MARECHALERIE VINCENT')
('485278113', 'BRASSERIE LA KREUZE', "L'AVALEE DE LA KREUZE BRASSERIE LA KREUZE ADELINE BEAUJOIN CREATIONS")
('791550742', 'CARPTA', 'CARPTA LA MAISON DU PLOMB SERVICE MAISON')
('792426116', 'AUX DELICES', 'AUX DELICES TAIS PECHE WESTERN TIR')


In [305]:
for dep in all_deps:
    start_time = time.time()
    print(dep)
    # Add etab count
    cursor.execute(f'''INSERT INTO count_etab (siren, count) SELECT siren, count(*) as count FROM {f"siret_{dep}"} GROUP BY siren;''')
    stats()
    stats()

23
--- 0.06482815742492676 seconds ---
used memory : 17.0Go
--- 0.06510686874389648 seconds ---
used memory : 17.0Go


In [307]:
for row in cursor.execute('SELECT * FROM count_etab ORDER BY count DESC LIMIT 10;'):
    print(row)

('356000745', 194)
('356000000', 70)
('501542021', 66)
('538160656', 56)
('750401176', 52)
('262324700', 50)
('538160524', 49)
('495105918', 48)
('527809339', 41)
('172302119', 40)


In [312]:
for row in cursor.execute('''select ul.siren, tbl1.sum from unite_legale ul LEFT JOIN (select siren, SUM(count) as sum FROM count_etab GROUP BY siren) tbl1 ON tbl1.siren = ul.siren LIMIT 20'''):
    print(row)

('000325175', None)
('001807254', None)
('005410220', None)
('005410345', None)
('005410394', None)
('005410428', None)
('005410436', None)
('005410485', None)
('005410493', None)
('005410501', None)
('005410519', None)
('005410527', None)
('005410543', None)
('005410642', None)
('005410667', None)
('005410675', None)
('005410717', None)
('005410733', None)
('005410741', None)
('005410758', None)


## Nombre d'établissements et nombre d'établissements ouverts

In [ ]:
add_nombre_etab = (''' SELECT siren, count
                        GROUP BY siren

In [ ]:
add_nombre_etab = '''ALTER TABLE siret_23 ADD COLUMN nombre_etablissementsss '''

In [ ]:
cursor.execute(add_nombre_etab)

In [ ]:
start_time = time.time()
cursor.execute('''
DROP TABLE IF EXISTS count_etab
''')
cursor.execute('''
CREATE TABLE count_etab AS 
SELECT siren, count(siret) AS nombre_etab
FROM siret_23
GROUP BY siren''')
stats()

In [ ]:
for row in cursor.execute('''PRAGMA table_info(count_etab)'''):
    print (row)

In [ ]:
for row in cursor.execute('SELECT * FROM count_etab LIMIT 10;'):
    print(row)

In [ ]:
start_time = time.time()
cursor.execute('''UPDATE siret_23 AS dep SET nombre_etablissements = (SELECT nombre_etab
                                                                FROM siret_23 as dep INNER JOIN count_etab as etab
                                                                ON dep.siren = etab.siren
                                                                where siren=dep.siren)
                                                                  ''')
stats()

In [ ]:
select siren, count(*) as count_etab FROM siret_23 GROUP BY siren limit 10

In [ ]:
start_time = time.time()
cursor.execute('''UPDATE siret_23 SET nombre_etablissements = (select count(*) as count_etab FROM siret_23 GROUP BY siren)
                                                                  ''')
stats()

In [ ]:
start_time = time.time()
for row in cursor.execute('''SELECT nombre_etab FROM siret_23 as dep INNER JOIN count_etab as etab ON dep.siren = etab.siren'''):
    print (row)
stats()

In [ ]:
for row in cursor.execute('SELECT * FROM siret_23 LIMIT 10;'):
    print(row)

In [ ]:
cursor.execute('''SELECT siren, count(siret)
FROM siret_23
GROUp BY siren

JOIN siret_23 AD TAB WHERE TAB.siren = 

In [ ]:
for row in cursor.execute('''SELECT siren, count(siret) FROM siret_23 GROUP BY siren LIMIT 100'''):
    print(row)

In [ ]:
for row in cursor.execute('''SELECT * FROM siret_23 WHERE siren="172302119"'''):
    print(row)